<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 100px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Intelligence Data Science and Artificial Intelligence (IDEAI)</p>
  </div>

</div>

# **Convolutional Neural Network (CNN)**

## 1. Qué vamos a hacer en este notebook

Aquí ya no nos quedamos en una CNN abstracta.
Vamos a trabajar con un **dataset real de imágenes** para que la sesión tenga una base práctica auténtica.

Usaremos **CIFAR-10**, un dataset clásico de visión por computador.

## 2. Qué es CIFAR-10

CIFAR-10 contiene 60.000 imágenes de tamaño 32x32 en color, repartidas en 10 clases:

- airplane
- automobile
- bird
- cat
- deer
- dog
- frog
- horse
- ship
- truck

## 3. Por qué es útil para docencia

Aunque no es un dataset bancario, tiene varias ventajas:
- es estándar,
- permite explicar bien la lógica de las CNN,
- es ligero,
- y evita problemas de permisos o privacidad.

Una vez que el alumno entiende CIFAR-10, ya puede trasladar la idea a:
- documentos,
- firmas,
- DNI,
- facturas,
- o imágenes de cheques.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test :", x_test.shape)
print("y_test :", y_test.shape)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

## 4. Visualización inicial

Antes de entrenar nada, siempre conviene mirar ejemplos reales.
Esto ayuda a:
- detectar errores de carga,
- entender la variabilidad del dataset,
- y hacer la clase más intuitiva.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(10, 6))

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(class_names[int(y_train[i])], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Preprocesado de imágenes

Las imágenes llegan como enteros entre 0 y 255.
Para entrenar una red neuronal es habitual escalar a [0, 1].

Además, en CIFAR-10 la variable objetivo viene como columna, así que la aplanamos con `ravel()`.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = y_train.ravel()
y_test = y_test.ravel()

print(np.min(x_train), np.max(x_train))
print(y_train[:10])

## 6. Qué es una convolución

Una convolución consiste en aplicar un filtro pequeño sobre una imagen para detectar patrones locales.

### Intuición
Un filtro puede aprender a detectar:
- bordes verticales,
- bordes horizontales,
- texturas,
- esquinas,
- patrones repetidos.

### Jerarquía de aprendizaje
Una CNN suele aprender:

- primeras capas → bordes y texturas
- capas intermedias → partes de objetos
- capas profundas → objetos completos

In [ ]:
# Ejemplo conceptual de convolución 3x3 sobre una matriz pequeña
imagen = np.array([
    [1, 1, 1],
    [0, 0, 0],
    [1, 1, 1]
])

kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
])

resultado = np.sum(imagen * kernel)
print("Resultado de la convolución conceptual:", resultado)

## 7. Arquitectura CNN base

Empezamos con una CNN sencilla pero razonable:

- Conv2D
- MaxPooling
- Conv2D
- MaxPooling
- Flatten
- Dense
- Dropout
- Dense final

Esto permite enseñar la arquitectura clásica sin hacerla excesivamente pesada.

In [ ]:
cnn = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(10, activation="softmax")
])

cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.summary()

## 8. Entrenamiento

En una sesión docente suele bastar con pocas épocas para ver la lógica.
Aquí dejamos `epochs=10` como base razonable.

In [ ]:
history = cnn.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    verbose=1
)

## 9. Evolución del entrenamiento

Mirar las curvas es crucial para detectar:
- aprendizaje insuficiente,
- overfitting,
- o entrenamiento estable.

In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df.head())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_df["loss"], label="train_loss")
ax.plot(history_df["val_loss"], label="val_loss")
ax.set_title("Evolución de la pérdida")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_df["accuracy"], label="train_acc")
ax.plot(history_df["val_accuracy"], label="val_acc")
ax.set_title("Evolución de la accuracy")
ax.legend()
plt.show()

## 10. Evaluación final

Ahora evaluamos sobre test.

In [ ]:
test_loss, test_acc = cnn.evaluate(x_test, y_test, verbose=0)
print("Test loss    :", test_loss)
print("Test accuracy:", test_acc)

## 11. Predicciones de ejemplo

Mostrar predicciones ayuda muchísimo a que el alumno conecte el modelo con el resultado.

In [ ]:
pred_proba = cnn.predict(x_test[:10], verbose=0)
pred_class = np.argmax(pred_proba, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i])
    ax.set_title(f"Real: {class_names[y_test[i]]}\nPred: {class_names[pred_class[i]]}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 12. Qué enseñar aquí como profesor

Con este ejemplo puedes explicar perfectamente:

- qué hace una CNN,
- por qué las convoluciones son locales,
- qué hace pooling,
- qué significa overfitting en visión,
- y cómo pasar de una CNN simple a una más potente.

## 13. Traslado a banca

Aunque CIFAR-10 no sea bancario, el patrón metodológico es transferible a:
- clasificación de documentos,
- detección de firmas,
- detección de documentos manipulados,
- validación visual de formularios,
- análisis de imágenes de justificantes.

## 14. Ejercicios propuestos

1. Añade una tercera capa convolucional.  
2. Cambia el número de filtros de la segunda capa.  
3. Añade `BatchNormalization`.  
4. Cambia `Dropout(0.30)` por `Dropout(0.50)` y compara.  
5. Representa 20 imágenes mal clasificadas.  
6. Cambia el optimizador a `RMSprop`.  
7. Aumenta a 15 épocas y analiza si aparece overfitting.  
8. Calcula una matriz de confusión para CIFAR-10.  
9. Explica la diferencia entre `softmax` y `sigmoid`.  
10. Propón un caso bancario real donde usarías una CNN.

In [ ]:
# Escribe aquí tu solución

## 15. Solucionario orientativo

- tercera capa convolucional: aumenta capacidad representacional
- más filtros: más complejidad y mayor coste
- batch normalization: estabiliza entrenamiento
- dropout alto: puede reducir overfitting pero también infraajustar
- `softmax` se usa en multiclase; `sigmoid`, típicamente en binaria
- caso bancario real: clasificación de documentos o detección de manipulación visual